In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rc('text', usetex=True)    #Remove if system does not have LaTex installed
plt.rc('font', family='serif') #Remove if system does not have LaTex installed

# The Sky Model Line

In [ ]:
def SkyFlux(E, A_DM, E_line, sig_DM, A_C, E_Cline, sig_C, B1, B2, B3, P1, P2, P3, FOV): 
#B1,B2,and B3 are normalization constants for CXB broken power law with corresponding powers P1,P2,P3
    
    Flux_DM = A_DM * np.exp(-(E-E_line)**2 / (2*sig_DM**2))

    Flux_Coma = A_C * np.exp(-(E-E_Cline)**2 / (2*sig_C**2))

    Flux_CXB = np.where(
        E < 0.4,
        B1 * E**(-P1),
        np.where(
            E < 1.2,
            B2 * E**(-P2),
            B3 * E**(-P3)
        )
    )

    Flux_tot = FOV*(Flux_DM + Flux_Coma + Flux_CXB)
    return Flux_tot

In [ ]:
E = np.linspace(1, 10, 1000)
dE = E[1] - E[0]

In [ ]:
params = dict(
   #Sky FLux parameters
    A_DM = 8,
    E_line = 2,
    sig_DM = 2*(1/1500),
    A_C = 8,
    E_Cline = 3,
    sig_C = 3*(1/300),
    B1 = 6.2,
    B2 = 8.2,
    B3 = 8.0,
    P1 = 1.9,
    P2 = 1.6,
    P3 = 1.45,
    FOV = 1
    
)
S = SkyFlux(E, **params)

**FOV x2 Line**

In [ ]:
FOV2_params = dict(params)
FOV2_params["FOV"] = 2
S_FOV2 = SkyFlux(E, **FOV2_params)

**CXB Line Only**

In [ ]:
cxb_params = dict(params)
cxb_params["A_DM"] = 0.0  
cxb_params["A_C"]  = 0.0 
S_cxb = SkyFlux(E, **cxb_params)

# Window Function/Convolution

In [ ]:
def gaussian_filter_1d(signal, radius):
    """
    1D Gaussian convolution

    Parameters
    ----------
    signal : 1D numpy array
        Input sky model
    radius : float
        Gaussian sigma

    Returns
    -------
    blurred : 1D numpy array
    """
    n = len(signal)
    blurred = np.zeros(n)

    size = min(len(signal)//2, int(4 * radius))                 # kernel half-width
    #size = 7000
    sigma2 = radius**2

    for x in range(n):

        total_sum = 0.0
        pixel_val = 0.0

        start_i = max(0, x - size)
        end_i   = min(n, x + size + 1)

        for i in range(start_i, end_i):
            factor = np.exp(-(i-x)**2/(2*sigma2))
            total_sum += factor
            pixel_val += factor * signal[i]

        blurred[x] = pixel_val / total_sum

    return blurred

# Convolved (Observed) Line

In [ ]:
FWHM = 0.2
sigma_bins = FWHM/(2*np.sqrt(2*np.log(2))) / dE
Observed = gaussian_filter_1d(S, sigma_bins)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.semilogy(E, E*S , linewidth=2, label='Sky Model')
ax.semilogy(E, E*Observed , linewidth=2, label='Convolved (observed)')
ax.grid(True, which='both', alpha=0.3)
ax.set_xlim(1, 10)
ax.set_ylim(1, 100)
plt.legend()
plt.xlabel('Energy (keV)')
plt.ylabel('E x N(E)')
plt.show()

# Convolved (x2 $\Omega$)

In [ ]:
FWHM = 0.2
sigma_bins = FWHM/(2*np.sqrt(2*np.log(2))) / dE
Observed = gaussian_filter_1d(S, sigma_bins)
FWHM2 = 0.2
sigma_bins2 = FWHM2/(2*np.sqrt(2*np.log(2))) / dE
Observed2 = gaussian_filter_1d(S_FOV2, sigma_bins2)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.semilogy(E, E*S , linewidth=2, label='Sky Model')
ax.semilogy(E, E*Observed , linewidth=2, label='Convolved (observed)')
ax.semilogy(E, E*Observed2 , linewidth=2, linestyle='--', color='magenta', label='Convolved (x2 $\Omega$)')
ax.grid(True, which='both', alpha=0.3)
ax.set_xlim(1, 10)
ax.set_ylim(1, 100)
plt.legend()
plt.xlabel('Energy (keV)')
plt.ylabel('E x N(E)')
plt.show()

# Convolved (x0.1 $\sigma_{\mathrm{det}}$)

In [ ]:
FWHM = 0.2
sigma_bins = FWHM/(2*np.sqrt(2*np.log(2))) / dE
Observed = gaussian_filter_1d(S, sigma_bins)
FWHM2 = 0.02
sigma_bins2 = FWHM2/(2*np.sqrt(2*np.log(2))) / dE
Observed2 = gaussian_filter_1d(S, sigma_bins2)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.semilogy(E, E*S , linewidth=2, label='Sky Model')
ax.semilogy(E, E*Observed , linewidth=2, label='Convolved (observed)')
ax.semilogy(E, E*Observed2 , linewidth=2, linestyle='--', color='magenta', label='Convolved (x0.1 $\sigma_{\mathrm{det}}$)')
ax.grid(True, which='both', alpha=0.3)
ax.set_xlim(1, 10)
ax.set_ylim(1, 100)
plt.legend()
plt.xlabel('Energy (keV)')
plt.ylabel('E x N(E)')
plt.show()